In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import PolynomialFeatures

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

print('='*60)
print('🏆 Gold Strike - Customer Churn Predictor')
print('5-Model Stacking Ensemble | Target: 0.915+')
print('='*60)

# ==================================================
# 1. 自动找到文件路径
# ==================================================
print('\n📁 扫描数据...')
train_path = None
test_path = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        if 'train.csv' in filename:
            train_path = full_path
            print(f'找到训练集: {train_path}')
        elif 'test.csv' in filename:
            test_path = full_path
            print(f'找到测试集: {test_path}')

if train_path is None or test_path is None:
    raise FileNotFoundError('没有找到train.csv或test.csv')

# 加载数据
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f'训练集: {train.shape} | 测试集: {test.shape}')

id_col = 'id'
target_col = 'Churn'
train[target_col] = (train[target_col] == 'Yes').astype(int)
print(f'流失率: {train[target_col].mean():.2%}')

# ==================================================
# 2. 高级特征工程（修复 service_count 顺序）
# ==================================================
print('\n🔧 高级特征工程...')

def advanced_feature_engineering(df, is_train=True, train_df=None):
    df = df.copy()
    
    # 清洗TotalCharges
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    
    # 缺失值处理
    if is_train:
        total_charges_median = df['TotalCharges'].median()
        df['TotalCharges'].fillna(total_charges_median, inplace=True)
    else:
        total_charges_median = train_df['TotalCharges'].median()
        df['TotalCharges'].fillna(total_charges_median, inplace=True)
    
    # ✅ 先定义服务列表
    services = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 
                'OnlineBackup', 'DeviceProtection', 'TechSupport',
                'StreamingTV', 'StreamingMovies']
    
    # ✅ 在这里创建 service_count（必须在其他使用它的特征之前）
    df['service_count'] = df[services].isin(['Yes']).sum(axis=1)
    
    # ========== 基础特征 ==========
    df['avg_monthly'] = df['TotalCharges'] / (df['tenure'] + 1)
    df['charge_diff'] = df['TotalCharges'] - df['MonthlyCharges'] * df['tenure']
    df['monthly_per_service'] = df['MonthlyCharges'] / (df['service_count'] + 1)
    df['total_charges_log'] = np.log1p(df['TotalCharges'])
    df['tenure_log'] = np.log1p(df['tenure'])
    
    # ========== 交互特征 ==========
    df['high_value_new'] = ((df['MonthlyCharges'] > df['MonthlyCharges'].median()) & 
                            (df['tenure'] < 12)).astype(int)
    
    df['no_protection'] = ((df['OnlineSecurity'] == 'No') & 
                           (df['TechSupport'] == 'No')).astype(int)
    
    df['month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    
    df['contract_with_paperless'] = ((df['Contract'] == 'Month-to-month') & 
                                      (df['PaperlessBilling'] == 'Yes')).astype(int)
    
    # ========== 其他特征 ==========
    df['fiber_optic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    
    # ========== 比例特征 ==========
    df['monthly_to_total'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
    df['tenure_to_services'] = df['tenure'] / (df['service_count'] + 1)
    
    # ========== 分箱特征 ==========
    df['tenure_bin'] = pd.cut(df['tenure'], bins=[0, 6, 12, 24, 48, 100], 
                               labels=[0,1,2,3,4]).astype(int)
    df['monthly_bin'] = pd.qcut(df['MonthlyCharges'], q=5, 
                                 labels=[0,1,2,3,4], duplicates='drop').astype(int)
    
    # ========== 风险评分 ==========
    df['risk_score'] = (
        (df['Contract'] == 'Month-to-month') * 3 +
        (df['PaymentMethod'] == 'Electronic check') * 2 +
        (df['tenure'] < 12) * 2 +
        (df['MonthlyCharges'] > df['MonthlyCharges'].median()) * 1
    )
    
    # ========== 聚类特征 ==========
    cluster_features = df[['tenure', 'MonthlyCharges']].fillna(0)
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_predict(cluster_features)
    
    return df

# 应用特征工程
train = advanced_feature_engineering(train, is_train=True)
test = advanced_feature_engineering(test, is_train=False, train_df=train)

# ==================================================
# 3. 目标编码（Target Encoding）
# ==================================================
print('\n🎯 目标编码...')

def target_encode(train, test, cat_cols, target):
    train_encoded = train.copy()
    test_encoded = test.copy()
    
    for col in cat_cols:
        # 计算每个类别的目标均值
        target_means = train.groupby(col)[target].mean()
        
        # 平滑处理（防止过拟合）
        global_mean = train[target].mean()
        smoothing = 10
        counts = train.groupby(col)[target].count()
        target_means_smooth = (target_means * counts + global_mean * smoothing) / (counts + smoothing)
        
        train_encoded[f'{col}_te'] = train[col].map(target_means_smooth)
        test_encoded[f'{col}_te'] = test[col].map(target_means_smooth).fillna(global_mean)
    
    return train_encoded, test_encoded

# 选择高基数分类特征进行目标编码
high_card_cols = ['PaymentMethod', 'Contract', 'InternetService']
train, test = target_encode(train, test, high_card_cols, target_col)

# ==================================================
# 4. 多项式特征（修复重复列问题）
# ==================================================
print('\n📐 多项式特征...')

# 选择关键数值特征
key_num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'service_count', 'risk_score']
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_features = poly.fit_transform(train[key_num_cols])
poly_feat_names = poly.get_feature_names_out(key_num_cols)

# ✅ 重命名多项式特征，避免与原始列名冲突
poly_feat_names = [f'poly_{name}' for name in poly_feat_names]

train_poly = pd.DataFrame(poly_features, columns=poly_feat_names, index=train.index)
test_poly = pd.DataFrame(poly.transform(test[key_num_cols]), 
                         columns=poly_feat_names, index=test.index)

train = pd.concat([train, train_poly], axis=1)
test = pd.concat([test, test_poly], axis=1)

print(f'添加了 {len(poly_feat_names)} 个多项式特征')

# ==================================================
# 5. 特征预处理
# ==================================================
print('\n⚙️ 特征预处理...')

# 准备数据
X = train.drop(columns=[id_col, target_col])
y = train[target_col]
X_test = test.drop(columns=[id_col])
test_ids = test[id_col]

# 分类特征编码
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# 标准化
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# ✅ 删除重复列（放在X和X_test定义之后）
print(f'删除重复列前 - 训练集: {X.shape[1]} 特征')
X = X.loc[:, ~X.columns.duplicated()]
X_test = X_test.loc[:, ~X_test.columns.duplicated()]
print(f'删除重复列后 - 训练集: {X.shape[1]} 特征')

print(f'最终特征数: {X.shape[1]}')

# ==================================================
# 6. 优化后的模型参数（移除早停）
# ==================================================
print('\n🔥 配置模型...')

# LightGBM（无早停）
lgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.01,
    'max_depth': 8,
    'num_leaves': 64,
    'subsample': 0.75,
    'subsample_freq': 1,
    'colsample_bytree': 0.75,
    'reg_alpha': 0.15,
    'reg_lambda': 0.35,
    'min_child_samples': 30,
    'random_state': 42,
    'verbose': -1
}

# XGBoost（无早停）
xgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.01,
    'max_depth': 6,
    'min_child_weight': 3,
    'subsample': 0.75,
    'colsample_bytree': 0.75,
    'colsample_bylevel': 0.75,
    'reg_alpha': 0.15,
    'reg_lambda': 0.5,
    'gamma': 0.1,
    'random_state': 42,
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'verbosity': 0
}

# CatBoost（无早停）
cat_params = {
    'iterations': 1500,
    'learning_rate': 0.01,
    'depth': 7,
    'l2_leaf_reg': 7,
    'random_state': 42,
    'verbose': 0
}

# 神经网络
mlp_params = {
    'hidden_layer_sizes': (128, 64, 32),
    'activation': 'relu',
    'alpha': 0.0005,
    'batch_size': 256,
    'learning_rate': 'adaptive',
    'early_stopping': True,
    'validation_fraction': 0.1,
    'random_state': 42,
    'max_iter': 500,
    'verbose': False
}

# 随机森林
rf_params = {
    'n_estimators': 300,
    'max_depth': 12,
    'min_samples_split': 50,
    'min_samples_leaf': 20,
    'random_state': 42,
    'n_jobs': -1
}

# ==================================================
# 7. 多层Stacking集成（修复 predict_proba 问题）
# ==================================================
print('\n🏗️ 训练Stacking集成模型...')

# 第一层基模型
level0_models = [
    ('lgb', lgb.LGBMClassifier(**lgb_params)),
    ('xgb', xgb.XGBClassifier(**xgb_params)),
    ('cat', CatBoostClassifier(**cat_params)),
    ('rf', RandomForestClassifier(**rf_params)),
    ('mlp', MLPClassifier(**mlp_params))
]

# ✅ 第二层元模型：改用 LogisticRegression（支持 predict_proba）
level1_model = LogisticRegression(
    C=1.0, 
    max_iter=1000, 
    random_state=42,
    solver='lbfgs'
)

# 创建Stacking分类器
stacking_clf = StackingClassifier(
    estimators=level0_models,
    final_estimator=level1_model,
    cv=5,
    stack_method='predict_proba',  # ✅ 使用 predict_proba
    passthrough=False,
    n_jobs=-1
)

# 训练Stacking模型
print('训练中（这可能需要10-20分钟）...')
stacking_clf.fit(X, y)

# ==================================================
# 8. 交叉验证评估
# ==================================================
print('\n📊 交叉验证评估...')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    temp_model = lgb.LGBMClassifier(**lgb_params)
    temp_model.fit(X_tr, y_tr)
    val_pred = temp_model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_pred)
    cv_scores.append(auc)
    print(f'  Fold {fold+1}: {auc:.5f}')

print(f'\n✅ 平均CV分数: {np.mean(cv_scores):.5f} (±{np.std(cv_scores):.5f})')

# ==================================================
# 9. 预测测试集
# ==================================================
print('\n🔮 生成预测...')

# Stacking模型预测
test_preds = stacking_clf.predict_proba(X_test)[:, 1]

# 额外：用LightGBM单独预测作为补充
lgb_final = lgb.LGBMClassifier(**lgb_params)
lgb_final.fit(X, y)
lgb_preds = lgb_final.predict_proba(X_test)[:, 1]

# 加权融合
final_preds = test_preds * 0.7 + lgb_preds * 0.3

# ==================================================
# 10. 阈值优化
# ==================================================
print('\n🎯 阈值优化...')

# 用交叉验证找到最优阈值
val_preds_all = np.zeros(len(y))
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    temp_model = lgb.LGBMClassifier(**lgb_params)
    temp_model.fit(X_tr, y_tr)
    val_preds_all[val_idx] = temp_model.predict_proba(X_val)[:, 1]

best_threshold = 0.5
best_f1 = 0
for threshold in np.arange(0.3, 0.7, 0.01):
    y_pred_thresh = (val_preds_all > threshold).astype(int)
    f1 = f1_score(y, y_pred_thresh)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f'最优阈值: {best_threshold:.2f} | 最优F1: {best_f1:.4f}')

# ==================================================
# 11. 生成提交文件
# ==================================================
print('\n📝 生成提交文件...')

submission = pd.DataFrame({
    'id': test_ids,
    'Churn': final_preds
})
submission.to_csv('submission.csv', index=False)

print('✅ 提交文件已保存: submission.csv')
print('='*60)

# 显示下载链接
from IPython.display import FileLink
display(FileLink('submission.csv'))

# 打印特征重要性
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': lgb_final.feature_importances_
}).sort_values('importance', ascending=False).head(20)
print('\n📊 Top 20 重要特征:')
print(feature_importance.to_string(index=False))

🏆 Gold Strike - Customer Churn Predictor
5-Model Stacking Ensemble | Target: 0.915+

📁 扫描数据...
找到训练集: /kaggle/input/competitions/playground-series-s6e3/train.csv
找到测试集: /kaggle/input/competitions/playground-series-s6e3/test.csv
训练集: (594194, 21) | 测试集: (254655, 20)
流失率: 22.52%

🔧 高级特征工程...

🎯 目标编码...

📐 多项式特征...
添加了 15 个多项式特征

⚙️ 特征预处理...
删除重复列前 - 训练集: 55 特征
删除重复列后 - 训练集: 55 特征
最终特征数: 55

🔥 配置模型...

🏗️ 训练Stacking集成模型...
训练中（这可能需要10-20分钟）...

📊 交叉验证评估...
  Fold 1: 0.91606
  Fold 2: 0.91712
  Fold 3: 0.91660
  Fold 4: 0.91765
  Fold 5: 0.91480

✅ 平均CV分数: 0.91645 (±0.00098)

🔮 生成预测...

🎯 阈值优化...
最优阈值: 0.35 | 最优F1: 0.7022

📝 生成提交文件...
✅ 提交文件已保存: submission.csv


/kaggle/working/submission.csv


📊 Top 20 重要特征:
                          feature  importance
                      charge_diff       10659
                      avg_monthly        9068
              monthly_per_service        7831
                     TotalCharges        6691
                   MonthlyCharges        6634
                 monthly_to_total        6033
   poly_MonthlyCharges risk_score        5906
     poly_TotalCharges risk_score        5444
               tenure_to_services        5116
                total_charges_log        4736
                           tenure        4498
           poly_tenure risk_score        4345
poly_MonthlyCharges service_count        4138
       poly_tenure MonthlyCharges        4032
         poly_tenure TotalCharges        3788
  poly_TotalCharges service_count        3609
 poly_MonthlyCharges TotalCharges        3517
        poly_tenure service_count        3203
              poly_MonthlyCharges        1778
                    SeniorCitizen        1580
